# 1 - IMPORTAMOS LIBRERIAS

In [1]:
import torch
from torchvision import transforms
from PIL import Image

# 2 - CARGAMOS MODELO E IMAGENES PARA PROBAR

In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class CNN(nn.Module):

    def __init__(self):
        super(CNN, self).__init__()

        self.conv1 = nn.Conv2d(3,8,3,padding=1)
        self.conv2 = nn.Conv2d(8,16,3,padding=1)
        self.conv3 = nn.Conv2d(16,32,3,padding=1)

        self.fc1 = nn.Linear(32*28*28,64)
        self.fc2 = nn.Linear(64,2)

    def forward(self,x):

        x = F.relu(self.conv1(x))
        x = F.max_pool2d(x,2,2)

        x = F.relu(self.conv2(x))
        x = F.max_pool2d(x,2,2)

        x = F.relu(self.conv3(x))
        x = F.max_pool2d(x,2,2)

        x = torch.flatten(x,1)

        x = F.relu(self.fc1(x))
        x = self.fc2(x)

        return x

In [5]:
mi_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

mi_modelo = CNN().to(mi_device)

state_dict = torch.load("/content/model_neumonia.pth", map_location=mi_device)

mi_modelo.load_state_dict(state_dict)

mi_modelo.eval()

CNN(
  (conv1): Conv2d(3, 8, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(8, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv3): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (fc1): Linear(in_features=25088, out_features=64, bias=True)
  (fc2): Linear(in_features=64, out_features=2, bias=True)
)

In [6]:
def predict_image(image_path, model, class_names, device):
    """
    Predice la clase de una imagen usando un modelo entrenado.

    Args:
        image_path (str): Ruta de la imagen.
        model (torch.nn.Module): Modelo ya cargado y configurado para inferencia.
        class_names (list): Lista de nombres de clases.
        device (torch.device): Dispositivo donde está el modelo.

    Returns:
        str: Clase predicha.
    """
    # Preprocesamiento de la imagen
    preprocess = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    # Cargar y preprocesar la imagen
    image = Image.open(image_path).convert("RGB")
    image_tensor = preprocess(image).unsqueeze(0).to(device)  # Añadir dimensión batch y mover al dispositivo

    # Realizar predicción
    with torch.no_grad():
        outputs = model(image_tensor)
        _, preds = torch.max(outputs, 1)

    # Retornar la clase predicha
    return class_names[preds[0].item()]

In [7]:
class_names = ['NORMAL', 'PNEUMONIA']

In [8]:
img_normal = '/content/NORMAL2-IM-0058-0001.jpeg'
img_pneumonia = '/content/ryct.2020200034.fig5-day0.jpeg'

In [9]:
path_image = img_normal

predicted_class = predict_image(path_image, mi_modelo, class_names, mi_device)
print(f"La clase predicha es: {predicted_class}")

La clase predicha es: NORMAL


In [10]:
path_image = img_pneumonia

predicted_class = predict_image(path_image, mi_modelo, class_names, mi_device)
print(f"La clase predicha es: {predicted_class}")

La clase predicha es: PNEUMONIA
